# LEDGAR exploratory data analysis

Run `make eda` from the repository root before executing this notebook. The figures and overlap table below are precomputed by `src.eda`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display

root = Path.cwd().resolve()
if not (root / 'src').is_dir():
    root = root.parent
sys.path.insert(0, str(root))
from src.data import load_ledgar
from src.utils import load_json

stats = load_json(root / 'outputs/data/eda_stats.json')
label_names = list(load_ledgar()['train'].features['label'].names)
pd.set_option('display.max_colwidth', None)

## Dataset and overlap

In [ ]:
display(pd.DataFrame([{'split': split, 'clauses': size} for split, size in stats['split_sizes'].items()]))
display(pd.DataFrame([{'metric': 'train imbalance ratio', 'value': stats['train_labels']['imbalance_ratio']}, *[{'metric': key, 'value': value} for key, value in stats['overlap'].items()]]))
display(pd.DataFrame(stats['train_labels']['most_frequent']).assign(group='most').pipe(lambda x: pd.concat([x, pd.DataFrame(stats['train_labels']['least_frequent']).assign(group='least')], ignore_index=True)))

## Token lengths and test slices

In [ ]:
display(pd.DataFrame([{'tokenizer': name, 'split': split, **{key: value for key, value in values.items() if key != 'share_truncated'}, **{f'truncated >{limit}': share for limit, share in values['share_truncated'].items()}} for name, by_split in stats['tokenizers'].items() for split, values in by_split.items()]))
for slice_name, counts in stats['test_slices'].items():
    print(slice_name)
    display(pd.DataFrame(counts.items(), columns=[slice_name, 'test clauses']))

## Figures

In [ ]:
display(Image(filename=str(root / 'reports/figures/label_distribution.png')))
display(Image(filename=str(root / 'reports/figures/token_lengths.png')))

## Example clauses from the three rarest train labels

In [ ]:
ds = load_ledgar()
train = ds['train']
train_labels = train['label']
rarest = [item['label'] for item in stats['train_labels']['least_frequent'][:3]]
examples = []
for label_name in rarest:
    label_id = label_names.index(label_name)
    indices = (index for index, value in enumerate(train_labels) if value == label_id)
    for index in list(indices)[:3]:
        examples.append({'label': label_name, 'train_idx': index, 'clause': train[index]['text']})
display(pd.DataFrame(examples))

## Exact train/test duplicates

In [ ]:
overlap = pd.read_csv(root / 'outputs/data/test_overlap.csv')
exact = overlap.loc[overlap['exact_dup']].head(3)
display(pd.DataFrame([{'test_idx': int(row.test_idx), 'train_idx': int(row.nearest_train_idx), 'test clause': ds['test'][int(row.test_idx)]['text'], 'train twin': train[int(row.nearest_train_idx)]['text']} for row in exact.itertuples(index=False)]))